In [5]:
import sqlite3

class Category:
    def __init__(self, name):
        self.name = name

    def save(self, cursor):
        cursor.execute("INSERT INTO categories (name) VALUES (?)", (self.name,))

class Recipe:
    def __init__(self, name, ingredients, directions, category_id):
        self.name = name
        self.ingredients = ingredients
        self.directions = directions
        self.category_id = category_id

    def save(self, cursor):
        cursor.execute("INSERT INTO recipes (name, ingredients, directions, category_id) VALUES (?, ?, ?, ?)",
                       (self.name, self.ingredients, self.directions, self.category_id))

class RecipeOrganizerCLI:
    def __init__(self):
        self.conn = sqlite3.connect('recipes.db')
        self.c = self.conn.cursor()
        self.create_tables()

    def create_tables(self):
        self.c.execute('''CREATE TABLE IF NOT EXISTS categories (
                     id INTEGER PRIMARY KEY,
                     name TEXT)''')

        self.c.execute('''CREATE TABLE IF NOT EXISTS recipes (
                     id INTEGER PRIMARY KEY,
                     name TEXT,
                     ingredients TEXT,
                     directions TEXT,
                     category_id INTEGER,
                     FOREIGN KEY (category_id) REFERENCES categories(id))''')
        self.conn.commit()

    def add_recipe(self):
        name = input("Enter recipe name: ")
        ingredients = input("Enter ingredients (separated by commas): ")
        directions = input("Enter directions: ")
        category_name = input("Enter category name: ")

        # Validate input
        if not name or not ingredients or not directions or not category_name:
            print("Error: All fields are required.")
            return

        # Check if category exists, if not, create it
        self.c.execute("SELECT id FROM categories WHERE name=?", (category_name,))
        category_id = self.c.fetchone()
        if category_id is None:
            category = Category(category_name)
            category.save(self.c)
            category_id = self.c.lastrowid
        else:
            category_id = category_id[0]

        recipe = Recipe(name, ingredients, directions, category_id)
        recipe.save(self.c)
        self.conn.commit()
        print("Recipe added successfully.")

    def update_recipe(self):
        recipe_id = input("Enter the ID of the recipe you want to update: ")
        new_name = input("Enter the new name for the recipe: ")
        new_ingredients = input("Enter the new ingredients for the recipe (separated by commas): ")
        new_directions = input("Enter the new directions for the recipe: ")
        new_category_name = input("Enter the new category name for the recipe: ")

        # Validate input
        if not new_name or not new_ingredients or not new_directions or not new_category_name:
            print("Error: All fields are required.")
            return

        # Check if the new category exists, if not, create it
        self.c.execute("SELECT id FROM categories WHERE name=?", (new_category_name,))
        new_category_id = self.c.fetchone()
        if new_category_id is None:
            new_category = Category(new_category_name)
            new_category.save(self.c)
            new_category_id = self.c.lastrowid
        else:
            new_category_id = new_category_id[0]

        # Update the recipe in the database
        self.c.execute("UPDATE recipes SET name=?, ingredients=?, directions=?, category_id=? WHERE id=?",
                       (new_name, new_ingredients, new_directions, new_category_id, recipe_id))
        self.conn.commit()

        # Fetch the updated recipe from the database
        self.c.execute("SELECT * FROM recipes WHERE id=?", (recipe_id,))
        updated_recipe = self.c.fetchone()

        print("Recipe updated successfully.")
        print("Updated Recipe Details:")
        print("Name:", updated_recipe[1])
        print("Ingredients:", updated_recipe[2])
        print("Directions:", updated_recipe[3])
        print("Category ID:", updated_recipe[4])

    def delete_recipe(self):
        recipe_id = input("Enter the ID of the recipe you want to delete: ")
        # Delete the recipe from the database
        self.c.execute("DELETE FROM recipes WHERE id=?", (recipe_id,))
        self.conn.commit()
        print("Recipe deleted successfully.")

    def plan_meal(self):
        while True:
            print("\nPlan Meal")
            print("1. View Recipes by Category")
            print("2. View All Recipes")
            print("3. Exit")

            choice = input("Enter your choice: ")

            if choice == '1':
                self.view_recipes_by_category()
            elif choice == '2':
                self.view_all_recipes()
            elif choice == '3':
                print("Exiting...")
                break
            else:
                print("Invalid choice. Please try again.")

    def view_recipes_by_category(self):
        self.c.execute("SELECT * FROM categories")
        categories = self.c.fetchall()
        print("Categories:")
        for category in categories:
            print(f"{category[0]}. {category[1]}")

        category_id = input("Enter the ID of the category you want to view recipes for (or 0 to go back): ")
        if category_id == '0':
            return

        self.c.execute("SELECT * FROM recipes WHERE category_id=?", (category_id,))
        recipes = self.c.fetchall()
        if not recipes:
            print("No recipes found for this category.")
        else:
            print("Recipes:")
            for recipe in recipes:
                print("- Name:", recipe[1])
                print("  Ingredients:", recipe[2])
                print("  Directions:", recipe[3])

    def view_all_recipes(self):
        page_size = 5  # Number of recipes to display per page
        page_number = 1
        while True:
            offset = (page_number - 1) * page_size
            self.c.execute("SELECT * FROM recipes LIMIT ? OFFSET ?", (page_size, offset))
            recipes = self.c.fetchall()
            if not recipes:
                print("No more recipes to display.")
                return

            print("Recipes:")
            for recipe in recipes:
                print("- Name:", recipe[1])
                print("  Ingredients:", recipe[2])
                print("  Directions:", recipe[3])

            choice = input("Enter 'n' for next page, 'p' for previous page, or any other key to exit: ")
            if choice.lower() == 'n':
                page_number += 1
            elif choice.lower() == 'p':
                if page_number > 1:
                    page_number -= 1
                else:
                    print("You're already on the first page.")
            else:
                break


    def search_recipe(self):
        keyword = input("Enter keyword to search for recipe: ")
        self.c.execute("SELECT * FROM recipes WHERE name LIKE ? OR ingredients LIKE ? OR directions LIKE ?",
                       ('%' + keyword + '%', '%' + keyword + '%', '%' + keyword + '%'))
        recipes = self.c.fetchall()
        if not recipes:
            print("No matching recipes found.")
        else:
            for recipe in recipes:
                print("Name:", recipe[1])
                print("Ingredients:", recipe[2])
                print("Directions:", recipe[3])
                print("Category ID:", recipe[4])
                print()

    def view_categories(self):
        self.c.execute("SELECT categories.name, COUNT(recipes.id) AS num_recipes FROM categories LEFT JOIN recipes ON categories.id = recipes.category_id GROUP BY categories.id")
        categories = self.c.fetchall()
        for category in categories:
            print("Category:", category[0])
            print("Number of Recipes:", category[1])
            print()

    def export_recipes(self):
        with open("recipes_export.txt", "w") as file:
            self.c.execute("SELECT * FROM recipes")
            recipes = self.c.fetchall()
            for recipe in recipes:
                file.write(f"Name: {recipe[1]}\n")
                file.write(f"Ingredients: {recipe[2]}\n")
                file.write(f"Directions: {recipe[3]}\n")
                file.write("\n")
        print("Recipes exported successfully.")

    def display_menu(self):
        while True:
            print("\nRecipe Organizer")
            print("1. Add Recipe")
            print("2. Update Recipe")
            print("3. Delete Recipe")
            print("4. Plan Meal")
            print("5. Search Recipe")
            print("6. View All Categories")
            print("7. Export Recipes")
            print("8. Exit")

            choice = input("Enter your choice: ")

            if choice == '1':
                self.add_recipe()
            elif choice == '2':
                self.update_recipe()
            elif choice == '3':
                self.delete_recipe()
            elif choice == '4':
                self.plan_meal()
            elif choice == '5':
                self.search_recipe()
            elif choice == '6':
                self.view_categories()
            elif choice == '7':
                self.export_recipes()
            elif choice == '8':
                print("Exiting...")
                break
            else:
                print("Invalid choice. Please try again.")

    def close_connection(self):
        self.conn.close()

if __name__ == "__main__":
    organizer = RecipeOrganizerCLI()
    organizer.display_menu()
    organizer.close_connection()



Recipe Organizer
1. Add Recipe
2. Update Recipe
3. Delete Recipe
4. Plan Meal
5. Search Recipe
6. View All Categories
7. Export Recipes
8. Exit
Enter your choice: 1
Enter recipe name: Pasta
Enter ingredients (separated by commas): Salt, Pepper, Meat, Pasta
Enter directions: Boil and Cook
Enter category name: Lunch
Recipe added successfully.

Recipe Organizer
1. Add Recipe
2. Update Recipe
3. Delete Recipe
4. Plan Meal
5. Search Recipe
6. View All Categories
7. Export Recipes
8. Exit
Enter your choice: 1
Enter recipe name: Spagetti
Enter ingredients (separated by commas): Sauce, Chicken
Enter directions: Boil and Cook
Enter category name: Dinner
Recipe added successfully.

Recipe Organizer
1. Add Recipe
2. Update Recipe
3. Delete Recipe
4. Plan Meal
5. Search Recipe
6. View All Categories
7. Export Recipes
8. Exit
Enter your choice: 4

Plan Meal
1. View Recipes by Category
2. View All Recipes
3. Exit
Enter your choice: 2
Recipes:
- Name: Pasta
  Ingredients: Salt, Pepper, Meat, Pasta
 